# FTD Euler Explicit coupling scheme.

## Introduction
By Guillaume BOIS, october 2024.

The test case works on the validation of the Front-Tracking problem, and specifically the coupling between various equation in a global explicit solver. To assess the validity of the solver for the equation system, a simple one-dimensional time-dependant problem is set. $x$ is taken as the discretised direction with $\Delta x=0.2$. 
Boundary conditions can be troublesome (especially when time-dependant) so x-periodic conditions are considered (the interface is away from this BC); other BC are symmetry. We assume a uniform source in the momentum equation. Diffusion (of momentum and temperature) is not considered.

### TRUST parameters
 
* Version TRUST : 1.9.4 
* "Type of problem: 2D Front Tracking" 
* "Discretization: VDF" 
* "Time scheme: Schema_euler_explicite" 
* Turbulence model: none 
* "Solving of equations: Navier_Stokes_FT_Disc, Transport_Interface_FT_Disc, Convection_diffusion_Temperature_FT_Disc" 


The timestep is prescribed at $\Delta t = 2s$ with an initial velocity $u^0=0.1$ to cross an integer number of cells per timestep (increasing with time, as the velocity). CFL stability will be broken by the velocity increase due to the momentum source so that we use a very large facsec (100). The front is planar and described by two markers (a large ideal length is used to avoid remeshing). VOF-like mass conservation is deactivated.

* Initial condition ($t=0$): 
\begin{align}
u^0 &= 0.1 \\
x^0_i& =0.5
\end{align}

* Governing Equations : 
\begin{align}
\nabla \cdot u &= 0 \\
\frac{\partial u}{\partial t}  &= \frac{S- \nabla P}{\rho} \\
x_i& =0.1
\end{align}
with $S=0.05si$. Density is uniformly set to 1. The interface velocity is obtained by interpolation of $u$ as $u_i = u(x_i)$.

* Boundary Conditions : 
Periodic on velocity.

Symmetry on FT markers (90° contact angle).

% And 2 dirichlet for $T_{of}$

The solution is a uniform velocity and pressure. Explicit temporal discretisation leads to :
\begin{equation}
u^{n+1} = u^n + \Delta t S / \rho + \nabla P / \rho
\end{equation}
where $\delta u = \Delta t S / \rho = 0.1\,m/s$. Because the source is uniform, the pressure projection leads to a constant pressure which then vanishes from the previous equation. Then, we have a constant acceleration : 
\begin{align}
u^0 &= 0.1 \\  
u^1 &= u^0+\delta u = 0.1+0.1 = 2 u^0 = 0.2\\  
u^2 &= u^1+\delta u = 3 u^0 = 0.3\\  
u^{n} &= (n+1) u^0 = (n+1) 0.1
\end{align}

Regarding the interface position (initially set to the first cell centre: $x_i =0.1$), we obtain a displacement proportional to $\Delta x = \Delta t u_0$ (by choice of parameters):
\begin{align}
x^0 &= 0.1 \\  
x^1 &= x^0+\Delta t u_0 = x^0+\Delta x = 0.3 \\  
x^2 &= x^1+\Delta t u_1 = x^0+\Delta x +2 \Delta x = x^0 + 3 \Delta x = 0.7 \\  
x^3 &= x^2+\Delta t u_2 = x^0+3\Delta x +3 \Delta x = x^0 + 6 \Delta x = 1.3 \\  
x^n &= x^0+ (1+2+3+\dots + n)\Delta x  = x_0 + n(n+1)/2 \Delta x
\end{align}



% Image de la config initiale (interf, norme vitesse et vec_ui)
## Verification
The solution

Mp
Ai
Distance
Indicatrice
\begin{equation}
T(x) = sin(\frac{2\pi}{L_x}(x-x_i))
\end{equation}


In [ ]:
import numpy
from trustutils import run
from pathlib import Path
import os
import shutil

## Fluid and flow properties

## Numerical parameters

In [ ]:
time_schemes = ["schema_euler_explicite", "RK3_FT", "Runge_Kutta_3_FT"]
facsec = 1
schemes = {"EFSTAB": "ef_stab { alpha 0.2 }",
           "MUSCL": "muscl",
           "UPWIND": "amont"}
turbmod = ["STD", "SST"]

meshes = {}
meshes["GAMBITFINE"] = {"geo_file": "dom-gambit-fine.geo", "mesh": "gambit-fine.geom"}
meshes["GAMBITCOAR"] = {"geo_file": "dom-gambit-coarse.geo", "mesh": "gambit-coarse.geom"}
meshes["GMSH"] = {"geo_file": "dom-gmsh.geo", "mesh": "gmsh.msh"}


def link_geo_and_mesh(thecase, thebuild, dest):
    if (thecase == "GAMBITFINE"):
        themesh = "gambitFine.geom"
        thegeo =  "dom-gambit-fine.geo"
    elif (thecase == "GAMBITCOAR"):
        themesh = "gambitCoarse.msh"
        thegeo = "dom-gambit-coarse.geo"
    elif (thecase == "GMSH"):
        themesh = "gmsh.geom"
        thegeo = "dom-gmsh.geo"
    else:
        print("Case must be GAMBITFINE, GAMBITCOAR or GMSH.")
        
    # Path(f"{dest}/{themesh}").symlink_to(Path(f"{thebuild}/MESHES/{themesh}"))
    Path(f"{dest}/dom.geo").symlink_to(Path(f"{thebuild}/MESHES/{thegeo}"))
    

In [ ]:
dsub = {}
# dsub["rho"] = rho
repo = "Eul"
run.reset()
run.addCaseFromTemplate("FTD_Euler_Explicit_Scheme.data",
                                        repo,
                                        dic=dsub)
                
run.printCases()

In [ ]:
run.runCases()

## Tests Description
 
Hydraulic initial conditions: fluid U = V = W = 0 m/s 
Hydraulic boundary condition: 
  *  The velocity is fixed in order to obtain Re = UD/$\nu$ = 17054 or 33720
  *  To create the Von Karman vortex street quicker, an asymmetry is created when initializing the velocity initial condition.
  *  CYLINDER paroi\_fixe 
  *  WALL paroi\_fixe 
  *  OUTLET frontiere\_ouverte\_pression\_imposee Champ\_Front\_Uniforme 1 0.0 
  *  INLET frontiere\_ouverte\_vitesse\_imposee Champ\_Fonc\_xyz dom 2 0.44/0.87 0.01*sin(x) 
 Turbulent boundary condition: 
  *  Turbulence kinetic energy is given by k=2/2(I*Uo$)^2$ where I is the turbulence intensity, taken to 1.5 \% (See experimental details page 2 of $[1]$) 
  *  Turbulence dissipation rate is given by {\Large $ \epsilon $} =Cmu$^{0.75}$*k$^{1.5}$/l where Cmu=0.09 and l turbulence scale length is taken to 0.07*2*R (See reason in $[2]$)
  *  CYLINDER paroi 
  *  WALL paroi 
  *  OUTLET frontiere\_ouverte K\_EPS\_EXT Champ\_Front\_Uniforme 2 k {\Large $ \epsilon $} 
  *  INLET  frontiere\_ouverte\_K\_eps\_impose Champ\_front\_Uniforme 2 k {\Large $ \epsilon $} 
 


### Physical properties
 
  



In [ ]:
from trustutils import visit
lata = "Eul/lata/post.lata"
#fig = visit.Show(lata,"Pseudocolor","INDICATRICE_INTERF_ELEM_dom", iteration=0, size=5, plotmesh=True,nY=2,nX=1,empty=True)
fig = visit.Show(lata,"Mesh","dom", iteration=0, size=5, plotmesh=True,nY=2,nX=1)
fig.addMesh("INTERFACES")
fig.meshColor("red")
fig.addField("Eul/lata/post.lata", "Vector", "VITESSE_SOM_INTERFACES",plotmesh=True)
fig.blackVector()
fig.visuOptions(["no_axes","no_bounding_box","no_triad"])
# Reproduction of visit commands
fig.visitCommand("SaveWindowAtts = SaveWindowAttributes()")
fig.visitCommand("SaveWindowAtts.width = 1024")
fig.visitCommand("SaveWindowAtts.height = 512")
# fig.visitCommand("SaveWindowAtts.resConstraint = ScreenProportions")
fig.visitCommand("SaveWindowAtts.subWindowAtts.win1.size = (128, 128)")
fig.visitCommand("SetSaveWindowAttributes(SaveWindowAtts)")
#
fig.visitCommand("VectorAtts = VectorAttributes()")
fig.visitCommand("VectorAtts.useStride = 1")
fig.visitCommand("VectorAtts.stride = 1")
fig.visitCommand("VectorAtts.autoScale = 1")
fig.visitCommand("VectorAtts.scaleByMagnitude = 1")
fig.visitCommand("VectorAtts.vectorColor = (153, 204, 200, 255)")
fig.visitCommand("VectorAtts.scale = 10.")
fig.visitCommand("VectorAtts.headSize = 0.5")
fig.visitCommand("VectorAtts.lineWidth = 5")
fig.visitCommand("VectorAtts.vectorOrigin = VectorAtts.Middle")
fig.visitCommand("SetPlotOptions(VectorAtts)")
fig.visitCommand("DrawPlots()")
# 
fig.add("Eul/lata/post.lata", "Pseudocolor", "VITESSE_FACES_dom_dual_magnitude",iteration=1,plotmesh=True, title="toto",xIndice=0,yIndice=1)
fig.addField("Eul/lata/post.lata", "Vector", "VITESSE_SOM_INTERFACES",plotmesh=True)
fig.addMesh("INTERFACES")
fig.meshColor("red")
#fig.addField("./grad_u_transpose_3d.lata", "Vector", "VITESSE_SOM_dom",plotmesh=True)


#visu.addField(plottype="vector", name="VITESSE_FACES_dom_dual")
fig.plot()

Detail of the mesh around the cylinder and Y+ values:



In [ ]:
from trustutils import visit
 
visu = visit.Show("0.44_muscl_Gambit/test.lata","Pseudocolor","Y_PLUS_ELEM_dom",mesh="dom")
visu.addField("0.44_muscl_Gambit/test.lata","Mesh","dom")
visu.zoom2D([-0.05,0.06,-0.05,0.05])
visu.visuOptions(["no_axes"])
visu.visuOptions(["no_bounding_box"])
visu.plot()

\newpage 
\textbf {B) Refined Gambit mesh} 
 It has $NUMBER\_OF\_ELEMENTS\_GambitFin$ cells.
 


In [ ]:
from trustutils import visit
 
visu = visit.Show("0.44_muscl_GambitFin/test.lata","Mesh","dom")
visu.plot()

Detail of the mesh around the cylinder and Y+ values:



In [ ]:
from trustutils import visit
 
visu = visit.Show("0.44_muscl_Gambit/test.lata","Pseudocolor","Y_PLUS_ELEM_dom",mesh="dom")
visu.addField("0.44_muscl_GambitFin/test.lata","Mesh","dom")
visu.zoom2D([-0.05,0.06,-0.05,0.05])
visu.visuOptions(["no_axes"])
visu.visuOptions(["no_bounding_box"])
visu.plot()

\newpage 
\textbf {C) Refined Gmsh mesh} 
 It has $NUMBER\_OF\_ELEMENTS\_Gmsh$ cells.
 


In [ ]:
from trustutils import visit
 
visu = visit.Show("0.44_muscl_Gmsh/test.lata","Mesh","dom")
visu.plot()

Detail of the mesh around the cylinder and Y+ values:



In [ ]:
from trustutils import visit
 
visu = visit.Show("0.44_muscl_Gambit/test.lata","Pseudocolor","Y_PLUS_ELEM_dom",mesh="dom")
visu.addField("0.44_muscl_Gmsh/test.lata","Mesh","dom")
visu.zoom2D([-0.05,0.06,-0.05,0.05])
visu.visuOptions(["no_axes"])
visu.visuOptions(["no_bounding_box"])
visu.plot()

\newpage 



## 2D Results
 
The Trio\_U simulations have been performed up to tmax=10*L/U in order to get sufficient cycles allowing the calculation of a frequency.

 


Flow in the experiment (extracted from $[1]$) :

![](src/flow.jpg)

\newpage 



### Scheme ef\_stab on mesh Gambit
 
 Velocity (Re=17054)



In [ ]:
from trustutils import visit
 
visu = visit.Show("0.44_ef_stab_Gambit/test.lata","Pseudocolor","norme_VITESSE_SOM_dom",mesh="dom")
visu.zoom2D([-0.25,1,-0.6,0.5])
visu.visuOptions(["no_axes"])
visu.visuOptions(["no_bounding_box"])
visu.plot()

Velocity (Re=33720)



In [ ]:
from trustutils import visit
 
visu = visit.Show("0.87_ef_stab_Gambit/test.lata","Pseudocolor","norme_VITESSE_SOM_dom",mesh="dom")
visu.zoom2D([-0.25,1,-0.6,0.5])
visu.visuOptions(["no_axes"])
visu.visuOptions(["no_bounding_box"])
visu.plot()

Zoom near the cylinder (Re=17054)



In [ ]:
from trustutils import visit
 
visu = visit.Show("0.44_ef_stab_Gambit/test.lata","Pseudocolor","norme_VITESSE_SOM_dom",mesh="dom")
visu.addField("0.44_ef_stab_Gambit/test.lata","Vector","VITESSE_SOM_dom",mesh="dom")
visu.slice(origin=[0.,0.,0.],normal=[0.,0.,1.],type_op="slice")
visu.zoom3D([0.21,0,16.7])
visu.visuOptions(["no_axes"])
visu.visuOptions(["no_bounding_box"])
visu.plot()

Zoom near the cylinder (Re=33720)



In [ ]:
from trustutils import visit
 
visu = visit.Show("0.87_ef_stab_Gambit/test.lata","Pseudocolor","norme_VITESSE_SOM_dom",mesh="dom")
visu.addField("0.87_ef_stab_Gambit/test.lata","Vector","VITESSE_SOM_dom",mesh="dom")
visu.slice(origin=[0.,0.,0.],normal=[0.,0.,1.],type_op="slice")
visu.zoom3D([0.21,0,16.7])
visu.visuOptions(["no_axes"])
visu.visuOptions(["no_bounding_box"])
visu.plot()

\newpage 



### Scheme muscl on mesh Gambit
 
 Velocity (Re=17054)



In [ ]:
from trustutils import visit
 
visu = visit.Show("0.44_muscl_Gambit/test.lata","Pseudocolor","norme_VITESSE_SOM_dom",mesh="dom")
visu.zoom2D([-0.25,1,-0.6,0.5])
visu.visuOptions(["no_axes"])
visu.visuOptions(["no_bounding_box"])
visu.plot()

Velocity (Re=33720)



In [ ]:
from trustutils import visit
 
visu = visit.Show("0.87_muscl_Gambit/test.lata","Pseudocolor","norme_VITESSE_SOM_dom",mesh="dom")
visu.zoom2D([-0.25,1,-0.6,0.5])
visu.visuOptions(["no_axes"])
visu.visuOptions(["no_bounding_box"])
visu.plot()

Zoom near the cylinder (Re=17054)



In [ ]:
from trustutils import visit
 
visu = visit.Show("0.44_muscl_Gambit/test.lata","Pseudocolor","norme_VITESSE_SOM_dom",mesh="dom")
visu.addField("0.44_muscl_Gambit/test.lata","Vector","VITESSE_SOM_dom",mesh="dom")
visu.slice(origin=[0.,0.,0.],normal=[0.,0.,1.],type_op="slice")
visu.zoom3D([0.21,0,16.7])
visu.visuOptions(["no_axes"])
visu.visuOptions(["no_bounding_box"])
visu.plot()

Zoom near the cylinder (Re=33720)



In [ ]:
from trustutils import visit
 
visu = visit.Show("0.87_muscl_Gambit/test.lata","Pseudocolor","norme_VITESSE_SOM_dom",mesh="dom")
visu.addField("0.87_muscl_Gambit/test.lata","Vector","VITESSE_SOM_dom",mesh="dom")
visu.slice(origin=[0.,0.,0.],normal=[0.,0.,1.],type_op="slice")
visu.zoom3D([0.21,0,16.7])
visu.visuOptions(["no_axes"])
visu.visuOptions(["no_bounding_box"])
visu.plot()

\newpage 



### Scheme amont on mesh Gambit
 
 Velocity (Re=17054)



In [ ]:
from trustutils import visit
 
visu = visit.Show("0.44_amont_Gambit/test.lata","Pseudocolor","norme_VITESSE_SOM_dom",mesh="dom")
visu.zoom2D([-0.25,1,-0.6,0.5])
visu.visuOptions(["no_axes"])
visu.visuOptions(["no_bounding_box"])
visu.plot()

Velocity (Re=33720)



In [ ]:
from trustutils import visit
 
visu = visit.Show("0.87_amont_Gambit/test.lata","Pseudocolor","norme_VITESSE_SOM_dom",mesh="dom")
visu.zoom2D([-0.25,1,-0.6,0.5])
visu.visuOptions(["no_axes"])
visu.visuOptions(["no_bounding_box"])
visu.plot()

Zoom near the cylinder (Re=17054)



In [ ]:
from trustutils import visit
 
visu = visit.Show("0.44_amont_Gambit/test.lata","Pseudocolor","norme_VITESSE_SOM_dom",mesh="dom")
visu.addField("0.44_amont_Gambit/test.lata","Vector","VITESSE_SOM_dom",mesh="dom")
visu.slice(origin=[0.,0.,0.],normal=[0.,0.,1.],type_op="slice")
visu.zoom3D([0.21,0,16.7])
visu.visuOptions(["no_axes"])
visu.visuOptions(["no_bounding_box"])
visu.plot()

Zoom near the cylinder (Re=33720)



In [ ]:
from trustutils import visit
 
visu = visit.Show("0.87_amont_Gambit/test.lata","Pseudocolor","norme_VITESSE_SOM_dom",mesh="dom")
visu.addField("0.87_amont_Gambit/test.lata","Vector","VITESSE_SOM_dom",mesh="dom")
visu.slice(origin=[0.,0.,0.],normal=[0.,0.,1.],type_op="slice")
visu.zoom3D([0.21,0,16.7])
visu.visuOptions(["no_axes"])
visu.visuOptions(["no_bounding_box"])
visu.plot()

\newpage 



### Scheme ef\_stab on mesh GambitFin
 
 Velocity (Re=17054)



In [ ]:
from trustutils import visit
 
visu = visit.Show("0.44_ef_stab_GambitFin/test.lata","Pseudocolor","norme_VITESSE_SOM_dom",mesh="dom")
visu.zoom2D([-0.25,1,-0.6,0.5])
visu.visuOptions(["no_axes"])
visu.visuOptions(["no_bounding_box"])
visu.plot()

Velocity (Re=33720)



In [ ]:
from trustutils import visit
 
visu = visit.Show("0.87_ef_stab_GambitFin/test.lata","Pseudocolor","norme_VITESSE_SOM_dom",mesh="dom")
visu.zoom2D([-0.25,1,-0.6,0.5])
visu.visuOptions(["no_axes"])
visu.visuOptions(["no_bounding_box"])
visu.plot()

Zoom near the cylinder (Re=17054)



In [ ]:
from trustutils import visit
 
visu = visit.Show("0.44_ef_stab_GambitFin/test.lata","Pseudocolor","norme_VITESSE_SOM_dom",mesh="dom")
visu.addField("0.44_ef_stab_GambitFin/test.lata","Vector","VITESSE_SOM_dom",mesh="dom")
visu.slice(origin=[0.,0.,0.],normal=[0.,0.,1.],type_op="slice")
visu.zoom3D([0.21,0,16.7])
visu.visuOptions(["no_axes"])
visu.visuOptions(["no_bounding_box"])
visu.plot()

Zoom near the cylinder (Re=33720)



In [ ]:
from trustutils import visit
 
visu = visit.Show("0.87_ef_stab_GambitFin/test.lata","Pseudocolor","norme_VITESSE_SOM_dom",mesh="dom")
visu.addField("0.87_ef_stab_GambitFin/test.lata","Vector","VITESSE_SOM_dom",mesh="dom")
visu.slice(origin=[0.,0.,0.],normal=[0.,0.,1.],type_op="slice")
visu.zoom3D([0.21,0,16.7])
visu.visuOptions(["no_axes"])
visu.visuOptions(["no_bounding_box"])
visu.plot()

\newpage 



### Scheme muscl on mesh GambitFin
 
 Velocity (Re=17054)



In [ ]:
from trustutils import visit
 
visu = visit.Show("0.44_muscl_GambitFin/test.lata","Pseudocolor","norme_VITESSE_SOM_dom",mesh="dom")
visu.zoom2D([-0.25,1,-0.6,0.5])
visu.visuOptions(["no_axes"])
visu.visuOptions(["no_bounding_box"])
visu.plot()

Velocity (Re=33720)



In [ ]:
from trustutils import visit
 
visu = visit.Show("0.87_muscl_GambitFin/test.lata","Pseudocolor","norme_VITESSE_SOM_dom",mesh="dom")
visu.zoom2D([-0.25,1,-0.6,0.5])
visu.visuOptions(["no_axes"])
visu.visuOptions(["no_bounding_box"])
visu.plot()

Zoom near the cylinder (Re=17054)



In [ ]:
from trustutils import visit
 
visu = visit.Show("0.44_muscl_GambitFin/test.lata","Pseudocolor","norme_VITESSE_SOM_dom",mesh="dom")
visu.addField("0.44_muscl_GambitFin/test.lata","Vector","VITESSE_SOM_dom",mesh="dom")
visu.slice(origin=[0.,0.,0.],normal=[0.,0.,1.],type_op="slice")
visu.zoom3D([0.21,0,16.7])
visu.visuOptions(["no_axes"])
visu.visuOptions(["no_bounding_box"])
visu.plot()

Zoom near the cylinder (Re=33720)



In [ ]:
from trustutils import visit
 
visu = visit.Show("0.87_muscl_GambitFin/test.lata","Pseudocolor","norme_VITESSE_SOM_dom",mesh="dom")
visu.addField("0.87_muscl_GambitFin/test.lata","Vector","VITESSE_SOM_dom",mesh="dom")
visu.slice(origin=[0.,0.,0.],normal=[0.,0.,1.],type_op="slice")
visu.zoom3D([0.21,0,16.7])
visu.visuOptions(["no_axes"])
visu.visuOptions(["no_bounding_box"])
visu.plot()

\newpage 



### Scheme amont on mesh GambitFin
 
 Velocity (Re=17054)



In [ ]:
from trustutils import visit
 
visu = visit.Show("0.44_amont_GambitFin/test.lata","Pseudocolor","norme_VITESSE_SOM_dom",mesh="dom")
visu.zoom2D([-0.25,1,-0.6,0.5])
visu.visuOptions(["no_axes"])
visu.visuOptions(["no_bounding_box"])
visu.plot()

Velocity (Re=33720)



In [ ]:
from trustutils import visit
 
visu = visit.Show("0.87_amont_GambitFin/test.lata","Pseudocolor","norme_VITESSE_SOM_dom",mesh="dom")
visu.zoom2D([-0.25,1,-0.6,0.5])
visu.visuOptions(["no_axes"])
visu.visuOptions(["no_bounding_box"])
visu.plot()

Zoom near the cylinder (Re=17054)



In [ ]:
from trustutils import visit
 
visu = visit.Show("0.44_amont_GambitFin/test.lata","Pseudocolor","norme_VITESSE_SOM_dom",mesh="dom")
visu.addField("0.44_amont_GambitFin/test.lata","Vector","VITESSE_SOM_dom",mesh="dom")
visu.slice(origin=[0.,0.,0.],normal=[0.,0.,1.],type_op="slice")
visu.zoom3D([0.21,0,16.7])
visu.visuOptions(["no_axes"])
visu.visuOptions(["no_bounding_box"])
visu.plot()

Zoom near the cylinder (Re=33720)



In [ ]:
from trustutils import visit
 
visu = visit.Show("0.87_amont_GambitFin/test.lata","Pseudocolor","norme_VITESSE_SOM_dom",mesh="dom")
visu.addField("0.87_amont_GambitFin/test.lata","Vector","VITESSE_SOM_dom",mesh="dom")
visu.slice(origin=[0.,0.,0.],normal=[0.,0.,1.],type_op="slice")
visu.zoom3D([0.21,0,16.7])
visu.visuOptions(["no_axes"])
visu.visuOptions(["no_bounding_box"])
visu.plot()

\newpage 



### Scheme ef\_stab on mesh Gmsh
 
 Velocity (Re=17054)



In [ ]:
from trustutils import visit
 
visu = visit.Show("0.44_ef_stab_Gmsh/test.lata","Pseudocolor","norme_VITESSE_SOM_dom",mesh="dom")
visu.zoom2D([-0.25,1,-0.6,0.5])
visu.visuOptions(["no_axes"])
visu.visuOptions(["no_bounding_box"])
visu.plot()

Velocity (Re=33720)



In [ ]:
from trustutils import visit
 
visu = visit.Show("0.87_ef_stab_Gmsh/test.lata","Pseudocolor","norme_VITESSE_SOM_dom",mesh="dom")
visu.zoom2D([-0.25,1,-0.6,0.5])
visu.visuOptions(["no_axes"])
visu.visuOptions(["no_bounding_box"])
visu.plot()

Zoom near the cylinder (Re=17054)



In [ ]:
from trustutils import visit
 
visu = visit.Show("0.44_ef_stab_Gmsh/test.lata","Pseudocolor","norme_VITESSE_SOM_dom",mesh="dom")
visu.addField("0.44_ef_stab_Gmsh/test.lata","Vector","VITESSE_SOM_dom",mesh="dom")
visu.slice(origin=[0.,0.,0.],normal=[0.,0.,1.],type_op="slice")
visu.zoom3D([0.21,0,16.7])
visu.visuOptions(["no_axes"])
visu.visuOptions(["no_bounding_box"])
visu.plot()

Zoom near the cylinder (Re=33720)



In [ ]:
from trustutils import visit
 
visu = visit.Show("0.87_ef_stab_Gmsh/test.lata","Pseudocolor","norme_VITESSE_SOM_dom",mesh="dom")
visu.addField("0.87_ef_stab_Gmsh/test.lata","Vector","VITESSE_SOM_dom",mesh="dom")
visu.slice(origin=[0.,0.,0.],normal=[0.,0.,1.],type_op="slice")
visu.zoom3D([0.21,0,16.7])
visu.visuOptions(["no_axes"])
visu.visuOptions(["no_bounding_box"])
visu.plot()

\newpage 



### Scheme muscl on mesh Gmsh
 
 Velocity (Re=17054)



In [ ]:
from trustutils import visit
 
visu = visit.Show("0.44_muscl_Gmsh/test.lata","Pseudocolor","norme_VITESSE_SOM_dom",mesh="dom")
visu.zoom2D([-0.25,1,-0.6,0.5])
visu.visuOptions(["no_axes"])
visu.visuOptions(["no_bounding_box"])
visu.plot()

Velocity (Re=33720)



In [ ]:
from trustutils import visit
 
visu = visit.Show("0.87_muscl_Gmsh/test.lata","Pseudocolor","norme_VITESSE_SOM_dom",mesh="dom")
visu.zoom2D([-0.25,1,-0.6,0.5])
visu.visuOptions(["no_axes"])
visu.visuOptions(["no_bounding_box"])
visu.plot()

Zoom near the cylinder (Re=17054)



In [ ]:
from trustutils import visit
 
visu = visit.Show("0.44_muscl_Gmsh/test.lata","Pseudocolor","norme_VITESSE_SOM_dom",mesh="dom")
visu.addField("0.44_muscl_Gmsh/test.lata","Vector","VITESSE_SOM_dom",mesh="dom")
visu.slice(origin=[0.,0.,0.],normal=[0.,0.,1.],type_op="slice")
visu.zoom3D([0.21,0,16.7])
visu.visuOptions(["no_axes"])
visu.visuOptions(["no_bounding_box"])
visu.plot()

Zoom near the cylinder (Re=33720)



In [ ]:
from trustutils import visit
 
visu = visit.Show("0.87_muscl_Gmsh/test.lata","Pseudocolor","norme_VITESSE_SOM_dom",mesh="dom")
visu.addField("0.87_muscl_Gmsh/test.lata","Vector","VITESSE_SOM_dom",mesh="dom")
visu.slice(origin=[0.,0.,0.],normal=[0.,0.,1.],type_op="slice")
visu.zoom3D([0.21,0,16.7])
visu.visuOptions(["no_axes"])
visu.visuOptions(["no_bounding_box"])
visu.plot()

\newpage 



### Scheme amont on mesh Gmsh
 
 Velocity (Re=17054)



In [ ]:
from trustutils import visit
 
visu = visit.Show("0.44_amont_Gmsh/test.lata","Pseudocolor","norme_VITESSE_SOM_dom",mesh="dom")
visu.zoom2D([-0.25,1,-0.6,0.5])
visu.visuOptions(["no_axes"])
visu.visuOptions(["no_bounding_box"])
visu.plot()

Velocity (Re=33720)



In [ ]:
from trustutils import visit
 
visu = visit.Show("0.87_amont_Gmsh/test.lata","Pseudocolor","norme_VITESSE_SOM_dom",mesh="dom")
visu.zoom2D([-0.25,1,-0.6,0.5])
visu.visuOptions(["no_axes"])
visu.visuOptions(["no_bounding_box"])
visu.plot()

Zoom near the cylinder (Re=17054)



In [ ]:
from trustutils import visit
 
visu = visit.Show("0.44_amont_Gmsh/test.lata","Pseudocolor","norme_VITESSE_SOM_dom",mesh="dom")
visu.addField("0.44_amont_Gmsh/test.lata","Vector","VITESSE_SOM_dom",mesh="dom")
visu.slice(origin=[0.,0.,0.],normal=[0.,0.,1.],type_op="slice")
visu.zoom3D([0.21,0,16.7])
visu.visuOptions(["no_axes"])
visu.visuOptions(["no_bounding_box"])
visu.plot()

Zoom near the cylinder (Re=33720)



In [ ]:
from trustutils import visit
 
visu = visit.Show("0.87_amont_Gmsh/test.lata","Pseudocolor","norme_VITESSE_SOM_dom",mesh="dom")
visu.addField("0.87_amont_Gmsh/test.lata","Vector","VITESSE_SOM_dom",mesh="dom")
visu.slice(origin=[0.,0.,0.],normal=[0.,0.,1.],type_op="slice")
visu.zoom3D([0.21,0,16.7])
visu.visuOptions(["no_axes"])
visu.visuOptions(["no_bounding_box"])
visu.plot()

\newpage 



## 1D Results
 
It can be seen that the beginning of the pseudo-steady state is at t=~5*L/U.
From this instant, an oscillation frequency can be determined.
 


### Scheme ef\_stab on mesh Gambit
 
 

In [ ]:
from trustutils import plot 
 
fig = plot.Graph(r"Pressure timetrace (0$<$time$<$10*L/U) at the cylinder surface for Re=17054") 
data = plot.loadText("0.44_ef_stab_Gambit/test_SONDE_PRESSION.son")
fig.add(1,2,label=r"(x,y)=(0,R)",marker='-')
data = plot.loadText("0.44_ef_stab_Gambit/test_SONDE_PRESSION.son")
fig.add(1,3,label=r"(x,y)=(0,-R)",marker='-')

fig.label(r"time (s)",r"Pressure (Pa)")

fig.visu(ymin=-0.5,ymax=0.0)


In [ ]:
from trustutils import plot 
 
fig = plot.Graph(r"Pressure timetrace (0$<$time$<$10*L/U) at the cylinder surface for Re=33720") 
data = plot.loadText("0.87_ef_stab_Gambit/test_SONDE_PRESSION.son")
fig.add(1,2,label=r"(x,y)=(0,R)",marker='-')
data = plot.loadText("0.87_ef_stab_Gambit/test_SONDE_PRESSION.son")
fig.add(1,3,label=r"(x,y)=(0,-R)",marker='-')

fig.label(r"time (s)",r"Pressure (Pa)")

fig.visu(ymin=-2.0,ymax=0.0)


In [ ]:
from trustutils import plot 
 
fig = plot.Graph(r"Drag and lift timetraces (9*L/U$<$time$<$10*L/U) on the cylinder for Re=17054") 
data = plot.loadText("0.44_ef_stab_Gambit/force.dat")
fig.add(1,2,label=r"drag",marker='-')
data = plot.loadText("0.44_ef_stab_Gambit/force.dat")
fig.add(1,3,label=r"lift",marker='-')

fig.label(r"time (s)",r"Force (N)")

fig.visu(ymin=-6.0,ymax=4.0)


In [ ]:
from trustutils import plot 
 
fig = plot.Graph(r"Drag and lift timetraces (9*L/U$<$time$<$10*L/U) on the cylinder for Re=33720") 
data = plot.loadText("0.87_ef_stab_Gambit/force.dat")
fig.add(1,2,label=r"drag",marker='-')
data = plot.loadText("0.87_ef_stab_Gambit/force.dat")
fig.add(1,3,label=r"lift",marker='-')

fig.label(r"time (s)",r"Force (N)")

fig.visu(ymin=-25.0,ymax=25.0)


### Scheme muscl on mesh Gambit
 
 

In [ ]:
from trustutils import plot 
 
fig = plot.Graph(r"Pressure timetrace (0$<$time$<$10*L/U) at the cylinder surface for Re=17054") 
data = plot.loadText("0.44_muscl_Gambit/test_SONDE_PRESSION.son")
fig.add(1,2,label=r"(x,y)=(0,R)",marker='-')
data = plot.loadText("0.44_muscl_Gambit/test_SONDE_PRESSION.son")
fig.add(1,3,label=r"(x,y)=(0,-R)",marker='-')

fig.label(r"time (s)",r"Pressure (Pa)")

fig.visu(ymin=-0.5,ymax=0.0)


In [ ]:
from trustutils import plot 
 
fig = plot.Graph(r"Pressure timetrace (0$<$time$<$10*L/U) at the cylinder surface for Re=33720") 
data = plot.loadText("0.87_muscl_Gambit/test_SONDE_PRESSION.son")
fig.add(1,2,label=r"(x,y)=(0,R)",marker='-')
data = plot.loadText("0.87_muscl_Gambit/test_SONDE_PRESSION.son")
fig.add(1,3,label=r"(x,y)=(0,-R)",marker='-')

fig.label(r"time (s)",r"Pressure (Pa)")

fig.visu(ymin=-2.0,ymax=0.0)


In [ ]:
from trustutils import plot 
 
fig = plot.Graph(r"Drag and lift timetraces (9*L/U$<$time$<$10*L/U) on the cylinder for Re=17054") 
data = plot.loadText("0.44_muscl_Gambit/force.dat")
fig.add(1,2,label=r"drag",marker='-')
data = plot.loadText("0.44_muscl_Gambit/force.dat")
fig.add(1,3,label=r"lift",marker='-')

fig.label(r"time (s)",r"Force (N)")

fig.visu(ymin=-6.0,ymax=4.0)


In [ ]:
from trustutils import plot 
 
fig = plot.Graph(r"Drag and lift timetraces (9*L/U$<$time$<$10*L/U) on the cylinder for Re=33720") 
data = plot.loadText("0.87_muscl_Gambit/force.dat")
fig.add(1,2,label=r"drag",marker='-')
data = plot.loadText("0.87_muscl_Gambit/force.dat")
fig.add(1,3,label=r"lift",marker='-')

fig.label(r"time (s)",r"Force (N)")

fig.visu(ymin=-25.0,ymax=25.0)


### Scheme amont on mesh Gambit
 
 

In [ ]:
from trustutils import plot 
 
fig = plot.Graph(r"Pressure timetrace (0$<$time$<$10*L/U) at the cylinder surface for Re=17054") 
data = plot.loadText("0.44_amont_Gambit/test_SONDE_PRESSION.son")
fig.add(1,2,label=r"(x,y)=(0,R)",marker='-')
data = plot.loadText("0.44_amont_Gambit/test_SONDE_PRESSION.son")
fig.add(1,3,label=r"(x,y)=(0,-R)",marker='-')

fig.label(r"time (s)",r"Pressure (Pa)")

fig.visu(ymin=-0.5,ymax=0.0)


In [ ]:
from trustutils import plot 
 
fig = plot.Graph(r"Pressure timetrace (0$<$time$<$10*L/U) at the cylinder surface for Re=33720") 
data = plot.loadText("0.87_amont_Gambit/test_SONDE_PRESSION.son")
fig.add(1,2,label=r"(x,y)=(0,R)",marker='-')
data = plot.loadText("0.87_amont_Gambit/test_SONDE_PRESSION.son")
fig.add(1,3,label=r"(x,y)=(0,-R)",marker='-')

fig.label(r"time (s)",r"Pressure (Pa)")

fig.visu(ymin=-2.0,ymax=0.0)


In [ ]:
from trustutils import plot 
 
fig = plot.Graph(r"Drag and lift timetraces (9*L/U$<$time$<$10*L/U) on the cylinder for Re=17054") 
data = plot.loadText("0.44_amont_Gambit/force.dat")
fig.add(1,2,label=r"drag",marker='-')
data = plot.loadText("0.44_amont_Gambit/force.dat")
fig.add(1,3,label=r"lift",marker='-')

fig.label(r"time (s)",r"Force (N)")

fig.visu(ymin=-6.0,ymax=4.0)


In [ ]:
from trustutils import plot 
 
fig = plot.Graph(r"Drag and lift timetraces (9*L/U$<$time$<$10*L/U) on the cylinder for Re=33720") 
data = plot.loadText("0.87_amont_Gambit/force.dat")
fig.add(1,2,label=r"drag",marker='-')
data = plot.loadText("0.87_amont_Gambit/force.dat")
fig.add(1,3,label=r"lift",marker='-')

fig.label(r"time (s)",r"Force (N)")

fig.visu(ymin=-25.0,ymax=25.0)


### Scheme ef\_stab on mesh GambitFin
 
 

In [ ]:
from trustutils import plot 
 
fig = plot.Graph(r"Pressure timetrace (0$<$time$<$10*L/U) at the cylinder surface for Re=17054") 
data = plot.loadText("0.44_ef_stab_GambitFin/test_SONDE_PRESSION.son")
fig.add(1,2,label=r"(x,y)=(0,R)",marker='-')
data = plot.loadText("0.44_ef_stab_GambitFin/test_SONDE_PRESSION.son")
fig.add(1,3,label=r"(x,y)=(0,-R)",marker='-')

fig.label(r"time (s)",r"Pressure (Pa)")

fig.visu(ymin=-0.5,ymax=0.0)


In [ ]:
from trustutils import plot 
 
fig = plot.Graph(r"Pressure timetrace (0$<$time$<$10*L/U) at the cylinder surface for Re=33720") 
data = plot.loadText("0.87_ef_stab_GambitFin/test_SONDE_PRESSION.son")
fig.add(1,2,label=r"(x,y)=(0,R)",marker='-')
data = plot.loadText("0.87_ef_stab_GambitFin/test_SONDE_PRESSION.son")
fig.add(1,3,label=r"(x,y)=(0,-R)",marker='-')

fig.label(r"time (s)",r"Pressure (Pa)")

fig.visu(ymin=-2.0,ymax=0.0)


In [ ]:
from trustutils import plot 
 
fig = plot.Graph(r"Drag and lift timetraces (9*L/U$<$time$<$10*L/U) on the cylinder for Re=17054") 
data = plot.loadText("0.44_ef_stab_GambitFin/force.dat")
fig.add(1,2,label=r"drag",marker='-')
data = plot.loadText("0.44_ef_stab_GambitFin/force.dat")
fig.add(1,3,label=r"lift",marker='-')

fig.label(r"time (s)",r"Force (N)")

fig.visu(ymin=-6.0,ymax=4.0)


In [ ]:
from trustutils import plot 
 
fig = plot.Graph(r"Drag and lift timetraces (9*L/U$<$time$<$10*L/U) on the cylinder for Re=33720") 
data = plot.loadText("0.87_ef_stab_GambitFin/force.dat")
fig.add(1,2,label=r"drag",marker='-')
data = plot.loadText("0.87_ef_stab_GambitFin/force.dat")
fig.add(1,3,label=r"lift",marker='-')

fig.label(r"time (s)",r"Force (N)")

fig.visu(ymin=-25.0,ymax=25.0)


### Scheme muscl on mesh GambitFin
 
 

In [ ]:
from trustutils import plot 
 
fig = plot.Graph(r"Pressure timetrace (0$<$time$<$10*L/U) at the cylinder surface for Re=17054") 
data = plot.loadText("0.44_muscl_GambitFin/test_SONDE_PRESSION.son")
fig.add(1,2,label=r"(x,y)=(0,R)",marker='-')
data = plot.loadText("0.44_muscl_GambitFin/test_SONDE_PRESSION.son")
fig.add(1,3,label=r"(x,y)=(0,-R)",marker='-')

fig.label(r"time (s)",r"Pressure (Pa)")

fig.visu(ymin=-0.5,ymax=0.0)


In [ ]:
from trustutils import plot 
 
fig = plot.Graph(r"Pressure timetrace (0$<$time$<$10*L/U) at the cylinder surface for Re=33720") 
data = plot.loadText("0.87_muscl_GambitFin/test_SONDE_PRESSION.son")
fig.add(1,2,label=r"(x,y)=(0,R)",marker='-')
data = plot.loadText("0.87_muscl_GambitFin/test_SONDE_PRESSION.son")
fig.add(1,3,label=r"(x,y)=(0,-R)",marker='-')

fig.label(r"time (s)",r"Pressure (Pa)")

fig.visu(ymin=-2.0,ymax=0.0)


In [ ]:
from trustutils import plot 
 
fig = plot.Graph(r"Drag and lift timetraces (9*L/U$<$time$<$10*L/U) on the cylinder for Re=17054") 
data = plot.loadText("0.44_muscl_GambitFin/force.dat")
fig.add(1,2,label=r"drag",marker='-')
data = plot.loadText("0.44_muscl_GambitFin/force.dat")
fig.add(1,3,label=r"lift",marker='-')

fig.label(r"time (s)",r"Force (N)")

fig.visu(ymin=-6.0,ymax=4.0)


In [ ]:
from trustutils import plot 
 
fig = plot.Graph(r"Drag and lift timetraces (9*L/U$<$time$<$10*L/U) on the cylinder for Re=33720") 
data = plot.loadText("0.87_muscl_GambitFin/force.dat")
fig.add(1,2,label=r"drag",marker='-')
data = plot.loadText("0.87_muscl_GambitFin/force.dat")
fig.add(1,3,label=r"lift",marker='-')

fig.label(r"time (s)",r"Force (N)")

fig.visu(ymin=-25.0,ymax=25.0)


### Scheme amont on mesh GambitFin
 
 

In [ ]:
from trustutils import plot 
 
fig = plot.Graph(r"Pressure timetrace (0$<$time$<$10*L/U) at the cylinder surface for Re=17054") 
data = plot.loadText("0.44_amont_GambitFin/test_SONDE_PRESSION.son")
fig.add(1,2,label=r"(x,y)=(0,R)",marker='-')
data = plot.loadText("0.44_amont_GambitFin/test_SONDE_PRESSION.son")
fig.add(1,3,label=r"(x,y)=(0,-R)",marker='-')

fig.label(r"time (s)",r"Pressure (Pa)")

fig.visu(ymin=-0.5,ymax=0.0)


In [ ]:
from trustutils import plot 
 
fig = plot.Graph(r"Pressure timetrace (0$<$time$<$10*L/U) at the cylinder surface for Re=33720") 
data = plot.loadText("0.87_amont_GambitFin/test_SONDE_PRESSION.son")
fig.add(1,2,label=r"(x,y)=(0,R)",marker='-')
data = plot.loadText("0.87_amont_GambitFin/test_SONDE_PRESSION.son")
fig.add(1,3,label=r"(x,y)=(0,-R)",marker='-')

fig.label(r"time (s)",r"Pressure (Pa)")

fig.visu(ymin=-2.0,ymax=0.0)


In [ ]:
from trustutils import plot 
 
fig = plot.Graph(r"Drag and lift timetraces (9*L/U$<$time$<$10*L/U) on the cylinder for Re=17054") 
data = plot.loadText("0.44_amont_GambitFin/force.dat")
fig.add(1,2,label=r"drag",marker='-')
data = plot.loadText("0.44_amont_GambitFin/force.dat")
fig.add(1,3,label=r"lift",marker='-')

fig.label(r"time (s)",r"Force (N)")

fig.visu(ymin=-6.0,ymax=4.0)


In [ ]:
from trustutils import plot 
 
fig = plot.Graph(r"Drag and lift timetraces (9*L/U$<$time$<$10*L/U) on the cylinder for Re=33720") 
data = plot.loadText("0.87_amont_GambitFin/force.dat")
fig.add(1,2,label=r"drag",marker='-')
data = plot.loadText("0.87_amont_GambitFin/force.dat")
fig.add(1,3,label=r"lift",marker='-')

fig.label(r"time (s)",r"Force (N)")

fig.visu(ymin=-25.0,ymax=25.0)


### Scheme ef\_stab on mesh Gmsh
 
 

In [ ]:
from trustutils import plot 
 
fig = plot.Graph(r"Pressure timetrace (0$<$time$<$10*L/U) at the cylinder surface for Re=17054") 
data = plot.loadText("0.44_ef_stab_Gmsh/test_SONDE_PRESSION.son")
fig.add(1,2,label=r"(x,y)=(0,R)",marker='-')
data = plot.loadText("0.44_ef_stab_Gmsh/test_SONDE_PRESSION.son")
fig.add(1,3,label=r"(x,y)=(0,-R)",marker='-')

fig.label(r"time (s)",r"Pressure (Pa)")

fig.visu(ymin=-0.5,ymax=0.0)


In [ ]:
from trustutils import plot 
 
fig = plot.Graph(r"Pressure timetrace (0$<$time$<$10*L/U) at the cylinder surface for Re=33720") 
data = plot.loadText("0.87_ef_stab_Gmsh/test_SONDE_PRESSION.son")
fig.add(1,2,label=r"(x,y)=(0,R)",marker='-')
data = plot.loadText("0.87_ef_stab_Gmsh/test_SONDE_PRESSION.son")
fig.add(1,3,label=r"(x,y)=(0,-R)",marker='-')

fig.label(r"time (s)",r"Pressure (Pa)")

fig.visu(ymin=-2.0,ymax=0.0)


In [ ]:
from trustutils import plot 
 
fig = plot.Graph(r"Drag and lift timetraces (9*L/U$<$time$<$10*L/U) on the cylinder for Re=17054") 
data = plot.loadText("0.44_ef_stab_Gmsh/force.dat")
fig.add(1,2,label=r"drag",marker='-')
data = plot.loadText("0.44_ef_stab_Gmsh/force.dat")
fig.add(1,3,label=r"lift",marker='-')

fig.label(r"time (s)",r"Force (N)")

fig.visu(ymin=-6.0,ymax=4.0)


In [ ]:
from trustutils import plot 
 
fig = plot.Graph(r"Drag and lift timetraces (9*L/U$<$time$<$10*L/U) on the cylinder for Re=33720") 
data = plot.loadText("0.87_ef_stab_Gmsh/force.dat")
fig.add(1,2,label=r"drag",marker='-')
data = plot.loadText("0.87_ef_stab_Gmsh/force.dat")
fig.add(1,3,label=r"lift",marker='-')

fig.label(r"time (s)",r"Force (N)")

fig.visu(ymin=-25.0,ymax=25.0)


### Scheme muscl on mesh Gmsh
 
 

In [ ]:
from trustutils import plot 
 
fig = plot.Graph(r"Pressure timetrace (0$<$time$<$10*L/U) at the cylinder surface for Re=17054") 
data = plot.loadText("0.44_muscl_Gmsh/test_SONDE_PRESSION.son")
fig.add(1,2,label=r"(x,y)=(0,R)",marker='-')
data = plot.loadText("0.44_muscl_Gmsh/test_SONDE_PRESSION.son")
fig.add(1,3,label=r"(x,y)=(0,-R)",marker='-')

fig.label(r"time (s)",r"Pressure (Pa)")

fig.visu(ymin=-0.5,ymax=0.0)


In [ ]:
from trustutils import plot 
 
fig = plot.Graph(r"Pressure timetrace (0$<$time$<$10*L/U) at the cylinder surface for Re=33720") 
data = plot.loadText("0.87_muscl_Gmsh/test_SONDE_PRESSION.son")
fig.add(1,2,label=r"(x,y)=(0,R)",marker='-')
data = plot.loadText("0.87_muscl_Gmsh/test_SONDE_PRESSION.son")
fig.add(1,3,label=r"(x,y)=(0,-R)",marker='-')

fig.label(r"time (s)",r"Pressure (Pa)")

fig.visu(ymin=-2.0,ymax=0.0)


In [ ]:
from trustutils import plot 
 
fig = plot.Graph(r"Drag and lift timetraces (9*L/U$<$time$<$10*L/U) on the cylinder for Re=17054") 
data = plot.loadText("0.44_muscl_Gmsh/force.dat")
fig.add(1,2,label=r"drag",marker='-')
data = plot.loadText("0.44_muscl_Gmsh/force.dat")
fig.add(1,3,label=r"lift",marker='-')

fig.label(r"time (s)",r"Force (N)")

fig.visu(ymin=-6.0,ymax=4.0)


In [ ]:
from trustutils import plot 
 
fig = plot.Graph(r"Drag and lift timetraces (9*L/U$<$time$<$10*L/U) on the cylinder for Re=33720") 
data = plot.loadText("0.87_muscl_Gmsh/force.dat")
fig.add(1,2,label=r"drag",marker='-')
data = plot.loadText("0.87_muscl_Gmsh/force.dat")
fig.add(1,3,label=r"lift",marker='-')

fig.label(r"time (s)",r"Force (N)")

fig.visu(ymin=-25.0,ymax=25.0)


### Scheme amont on mesh Gmsh
 
 

In [ ]:
from trustutils import plot 
 
fig = plot.Graph(r"Pressure timetrace (0$<$time$<$10*L/U) at the cylinder surface for Re=17054") 
data = plot.loadText("0.44_amont_Gmsh/test_SONDE_PRESSION.son")
fig.add(1,2,label=r"(x,y)=(0,R)",marker='-')
data = plot.loadText("0.44_amont_Gmsh/test_SONDE_PRESSION.son")
fig.add(1,3,label=r"(x,y)=(0,-R)",marker='-')

fig.label(r"time (s)",r"Pressure (Pa)")

fig.visu(ymin=-0.5,ymax=0.0)


In [ ]:
from trustutils import plot 
 
fig = plot.Graph(r"Pressure timetrace (0$<$time$<$10*L/U) at the cylinder surface for Re=33720") 
data = plot.loadText("0.87_amont_Gmsh/test_SONDE_PRESSION.son")
fig.add(1,2,label=r"(x,y)=(0,R)",marker='-')
data = plot.loadText("0.87_amont_Gmsh/test_SONDE_PRESSION.son")
fig.add(1,3,label=r"(x,y)=(0,-R)",marker='-')

fig.label(r"time (s)",r"Pressure (Pa)")

fig.visu(ymin=-2.0,ymax=0.0)


In [ ]:
from trustutils import plot 
 
fig = plot.Graph(r"Drag and lift timetraces (9*L/U$<$time$<$10*L/U) on the cylinder for Re=17054") 
data = plot.loadText("0.44_amont_Gmsh/force.dat")
fig.add(1,2,label=r"drag",marker='-')
data = plot.loadText("0.44_amont_Gmsh/force.dat")
fig.add(1,3,label=r"lift",marker='-')

fig.label(r"time (s)",r"Force (N)")

fig.visu(ymin=-6.0,ymax=4.0)


In [ ]:
from trustutils import plot 
 
fig = plot.Graph(r"Drag and lift timetraces (9*L/U$<$time$<$10*L/U) on the cylinder for Re=33720") 
data = plot.loadText("0.87_amont_Gmsh/force.dat")
fig.add(1,2,label=r"drag",marker='-')
data = plot.loadText("0.87_amont_Gmsh/force.dat")
fig.add(1,3,label=r"lift",marker='-')

fig.label(r"time (s)",r"Force (N)")

fig.visu(ymin=-25.0,ymax=25.0)


\newpage 



## Flow analysis
 
According schemes and meshes, Trio\_U predicts or not the Von Karman wake. The following tables summarize the wake state behind the cylinder at Re=17054 (the pattern are the same at Re=33720).



In [ ]:
from trustutils import plot 
 
columns=['EF\\_stab ', ' Muscl ', ' Upwind'] 
tab = plot.Table(columns)
tab.addLine([['Steady', 'Unsteady', 'Unsteady']],r"Gambit")
tab.addLine([['Unsteady', 'Unsteady', 'Steady']],r"GambitFin")
tab.addLine([['Steady', 'Unsteady', 'Steady']],r"Gmsh")
display(tab)

## Comparison between Trio\_U calculations and experiment
 
The steady drag coefficient number $C_{d}$ is defined as $C_{d}=\frac{Fx}{0.5*rho*U^2*D*W}$, where $Fx$ is the mean drag.
The dynamic lift coefficient number $C'_{l}$ is defined as $C'_{l}=\frac{F'y}{0.5*rho*U^2*D*W}$, where $F'y$ is the root mean square amplitude of the lift.
 The dominant frequency of dynamic lift (page 508 of $[1]$) allows the determination of the Strouhal
 number $S_{t}$ defined as $S_{t}=\frac{f.D}{U}$, where $f$ stands for the vortexes emission frequency.
 The following tables contain the Trio\_U calculated numbers for the three different schemes and meshes and the relative error versus the experimental values.
 


### Comparison of steady drag coefficient numbers
 
 Experimental value found in [1] at Re=17054 for $C_{d}$ is 1.10
Trio\_U calculated values are given below with relative error versus the experimental value in brackets:
 


In [ ]:
from trustutils import plot 
 
columns=['Mesh cells ', ' EF\\_stab ', ' Muscl ', ' Upwind'] 
tab = plot.Table(columns)
data = plot.loadText("Cd_0.44.dat",transpose=False, dtype="str", skiprows=0)
tab.setTitle("Comparison of steady drag coefficient numbers")
display(tab)

Experimental value found in [1] at Re=33720 for $C_{d}$ is 1.01
Trio\_U calculated values are given below with relative error versus the experimental value in brackets:
 


In [ ]:
from trustutils import plot 
 
columns=['Mesh cells ', ' EF\\_stab ', ' Muscl ', ' Upwind'] 
tab = plot.Table(columns)
data = plot.loadText("Cd_0.87.dat",transpose=False, dtype="str", skiprows=0)
display(tab)

### Comparison of dynamic lift coefficient numbers
 
 Experimental value found in [1] at Re=17054 for $C_{l}$ is 0.038 but reference [3] (page 462) gives value in the range 0.4-0.6 so relative error with Trio\_U is not computed:



In [ ]:
from trustutils import plot 
 
columns=['Mesh cells ', ' EF\\_stab ', ' Muscl ', ' Upwind'] 
tab = plot.Table(columns)
data = plot.loadText("Cl_0.44.dat",transpose=False, dtype="str", skiprows=0)
tab.setTitle("Comparison of dynamic lift coefficient numbers")
display(tab)

Experimental value found in [1] at Re=33720 for $C_{l}$ is 0.064 but reference [3] (page 462) gives value in the range 0.4-0.6 so relative error with Trio\_U is not computed:



In [ ]:
from trustutils import plot 
 
columns=['Mesh cells ', ' EF\\_stab ', ' Muscl ', ' Upwind'] 
tab = plot.Table(columns)
data = plot.loadText("Cl_0.87.dat",transpose=False, dtype="str", skiprows=0)
display(tab)

### Comparison of Strouhal numbers
 
 Experimental value found in [1] at Re=17054 for $S_{t}$ is 0.184
Trio\_U calculated values are given below with relative error versus the experimental value in brackets:
 


In [ ]:
from trustutils import plot 
 
columns=['Mesh cells ', ' EF\\_stab ', ' Muscl ', ' Upwind'] 
tab = plot.Table(columns)
data = plot.loadText("St_0.44.dat",transpose=False, dtype="str", skiprows=0)
tab.setTitle("Comparison of Strouhal numbers")
display(tab)

Experimental value found in [1] at Re=33720 for $S_{t}$ is 0.182
Trio\_U calculated values are given below with relative error versus the experimental value in brackets:
 


In [ ]:
from trustutils import plot 
 
columns=['Mesh cells ', ' EF\\_stab ', ' Muscl ', ' Upwind'] 
tab = plot.Table(columns)
data = plot.loadText("St_0.87.dat",transpose=False, dtype="str", skiprows=0)
display(tab)

\newpage 



## Conclusion
 
*  The results confirm one of the standard K-Eps model limitation which is to not predict accurately adverse
pressure gradient flow, overestimating the separation angle and thus generally underestimating the drag
 coefficient number.
 *  Upwind scheme sometimes does not predict Von Karman vortexes on every mesh, and even if it does
 the results are less good than the Muscl scheme.
 *  EF\_stab scheme has bad or false results on at least one mesh and further
 investigations will be mandatory to explain or fix this problem.
 *  Muscl scheme is the more robust scheme and despite of the standard K-Eps limitation give the more
 consistent results, except for the dynamic lift coefficient number.
 


## Recommendations for users
 
*  Correct K-Eps values for inlet boundary condition are critical to have an unstationary flow.
*  A small asymmetry is necessary to create quickly the Von Karman vortexes. This asymmetry may be created
 on the mesh or by a small perturbation on the initial condition for the velocity field (perhaps the last perturbation
 used for the calculations is not enough for the 2 meshes where upwind scheme resulted in a steady solution).
 *  Muscl scheme is the recommended scheme for this kind of calculation.
 *  For each scheme giving the Von Karman wake, the higher the number of mesh points on the cylinder, the better the results are.
 


## Computer performance
 


In [ ]:
run.tablePerf()